In [1]:
# pip install youtube-transcript-api

# step 1a- Indexing (Document Ingestion)

In [2]:
from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled

video_id = "Gfr50f6ZBvo"

try:
    api = YouTubeTranscriptApi()
    transcript_obj = api.fetch(video_id=video_id, languages=["en"])

    # Use .text attribute instead of ["text"]
    transcript = " ".join(snippet.text for snippet in transcript_obj)

    print(transcript)  # full concatenated text

except TranscriptsDisabled:
    print("No captions available for this video.")
except Exception as e:
    print(f"An unexpected error occurred: {e}")

An unexpected error occurred: 
Could not retrieve a transcript for the video https://www.youtube.com/watch?v=Gfr50f6ZBvo! This is most likely caused by:

YouTube is blocking requests from your IP. This usually is due to one of the following reasons:
- You have done too many requests and your IP has been blocked by YouTube
- You are doing requests from an IP belonging to a cloud provider (like AWS, Google Cloud Platform, Azure, etc.). Unfortunately, most IPs from cloud providers are blocked by YouTube.

There are two things you can do to work around this:
1. Use proxies to hide your IP address, as explained in the "Working around IP bans" section of the README (https://github.com/jdepoix/youtube-transcript-api?tab=readme-ov-file#working-around-ip-bans-requestblocked-or-ipblocked-exception).
2. (NOT RECOMMENDED) If you authenticate your requests using cookies, you will be able to continue doing requests for a while. However, YouTube will eventually permanently ban the account that you ha

In [3]:
transcript_obj

NameError: name 'transcript_obj' is not defined

In [ ]:
type(transcript_obj)

# Step b -Indexing(Text Splitting)

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [ ]:
splitter=RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
    )
chunks=splitter.create_documents([transcript])

In [ ]:
len(chunks)

In [ ]:
chunks[0]

# step 1c and 1d - Indexing(Embedding Generation and Storing in Vector Store)

In [ ]:
# embedding
from langchain_huggingface.embeddings import HuggingFaceEndpointEmbeddings
from dotenv import load_dotenv
import os

load_dotenv()
api_key = os.getenv("Hugging_face_api_token")
hf_embeddings_api = HuggingFaceEndpointEmbeddings(
    model="sentence-transformers/all-MiniLM-L6-v2",  # lightweight
    # model="Qwen/Qwen3-Embedding-8B",
    # task="feature-extraction",# heavyweight but uses a lot of quota token
    huggingfacehub_api_token=api_key,
)


In [ ]:
# pip install langchain-chroma

In [ ]:

#Depreciated
# from langchain_community.vectorstores import Chroma

#so use this
from langchain_chroma import Chroma

# Define persistent directory
persist_directory =os.path.abspath("./Kristal1_chroma_db")
# Create persistent Chroma vector store (SAVED TO DISK)
vector_store=Chroma.from_documents(
    documents=chunks,
    embedding=hf_embeddings_api,
    collection_name=f"yt_{video_id}",
    persist_directory=persist_directory # save to this folder in disk and not in RAM
)

print(f"Chroma vector store created with {vector_store._collection.count()} vectors")
print(f" Permanently saved to: {persist_directory}")
print(f" Collection name: yt_{video_id}")

In [ ]:
# # to remove the database perfectly
# import os
# import shutil

# # Remove the existing database directory
# persist_directory = "./Kristal_chroma_db"
# if os.path.exists(persist_directory):
#     print(f"Removing existing directory: {persist_directory}")
#     shutil.rmtree(persist_directory)
#     print("Directory removed successfully")
# else:
#     print("Directory does not exist")


In [ ]:
# Verify Persistence - Check Saved Files
import os
print(f"Files in {persist_directory}:")
if os.path.exists(persist_directory):
    for file in os.listdir(persist_directory):
        print(f"  📄 {file}")
else:
    print("  Directory not found yet")

In [ ]:
# Get Collection Information
print(f"📊 Collection stats:")
print(f"  - Name: {vector_store._collection.name}")
print(f"  - Documents: {vector_store._collection.count()}")

## Retrieve Document by ID (Chroma way)

In [ ]:
# Get all IDs from the collection
all_data = vector_store._collection.get(include=["embeddings","documents","metadatas"])

# Get the last ID
last_id = all_data['ids'][-1]
print(f"🔑 Last document ID: {last_id}")
    
    # Get document by ID
doc_result = vector_store._collection.get(ids=[last_id])
print(f"📝 Document content preview:")
print(doc_result['documents'][0])

In [ ]:
all_data

In [ ]:
doc_result

# Step-2:Retrieval

In [ ]:
retriever=vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={
        "k":4
    }
)

In [ ]:
retriever

In [ ]:
retriever.invoke("What is deepmind")

# STEP 3-Augmentation

In [ ]:
from langchain_core.prompts import PromptTemplate
prompt = PromptTemplate(
    template="""
      You are a helpful assistant.
      Answer ONLY from the provided transcript context.
      If the context is insufficient, just say you don't know.

      {context}
      Question: {question}
    """,
    input_variables = ['context', 'question']
)

In [ ]:
question="is the topic of aliens discussed in this video?if yes then what was discussed"
retrieved_docs=retriever.invoke(question)

In [ ]:
retrieved_docs

In [ ]:
context_text="\n\n".join(doc.page_content for doc in retrieved_docs)

In [ ]:
context_text

In [ ]:
final_prompt=prompt.invoke(
    {
        "context":context_text,
        "question":question
    }
)

In [ ]:
final_prompt

# Step 4 -Generation

In [ ]:
# pip install jupyter

# to download from huggingface using hf_hub_download

In [ ]:
# import os
# from huggingface_hub import hf_hub_download
# from langchain_community.llms import LlamaCpp

# # Set cache directory
# os.environ["HF_HOME"] = "/Users/kristalshrestha/Documents/Code/LLM_Scratch/models"

# # Download (only first time)
# model_path = hf_hub_download(
#     repo_id="hugging-quants/Llama-3.2-1B-Instruct-Q8_0-GGUF",
#     filename="llama-3.2-1b-instruct-q8_0.gguf"
# )

# print("Model path:", model_path)

# # Load model
# model = LlamaCpp(
#     model_path=model_path,
#     n_ctx=2048,
#     n_threads=8,
#     temperature=0.5
# )

# if u downloaded manually that gguf model from huggingface or somewhere else 

In [ ]:
from langchain_community.llms import LlamaCpp

model_path = "/Users/kristalshrestha/Documents/Code/LLM_Scratch/Langchain/6_RAG/5.1_Youtube_ChatBot_RAG_gguf/llama-3.2-1b-instruct-q8_0.gguf"

model = LlamaCpp(
    model_path=model_path,
    n_ctx=2048,
    n_threads=8,
    temperature=0.5
)


In [ ]:
answer=model.invoke(final_prompt)
print(answer.content)

In [ ]:
answer

# step 5- Chain

In [ ]:
from langchain_core.runnables import RunnableParallel,RunnablePassthrough,RunnableLambda
from langchain_core.output_parsers import StrOutputParser

In [ ]:
def format_docs(retrieved_docs):
    context_text="\n\n".join(doc.page_content for doc in retrieved_docs)
    return context_text

In [ ]:
parallel_chain=RunnableParallel(
    {
        "context":retriever|RunnableLambda(format_docs),
        "question":RunnablePassthrough()
    }
)

In [ ]:
parallel_chain.invoke("who is Demis")

In [ ]:
parser=StrOutputParser()

In [ ]:
main_chain=parallel_chain|prompt|model|parser

In [ ]:
main_chain.invoke("can you summarize the video")